# RAG Pipeline - Minimal Experiment


## 0. Setup & Environment


In [1]:
import sys
import os

# Reduce CUDA fragmentation (best applied before heavy CUDA allocations)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch

# --- Detect environment and set project root ---
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    # Clone repo (skip if already cloned)
    if not os.path.exists('/content/rag_project'):
        !git clone https://github.com/GanonthaBr/Robustness-of-RAG-systems-for-low-resources-langues.git /content/rag_project
    project_root = '/content/rag_project'

    # --- Fix dense_retriever.py on disk: cast to float32 before FAISS calls ---
    _retriever_path = os.path.join(project_root, 'retrieval', 'dense_retriever.py')
    with open(_retriever_path, 'r') as f:
        _src = f.read()
    _modified = False
    if 'np.vstack(embeddings)\n' in _src:
        _src = _src.replace('np.vstack(embeddings)\n', 'np.vstack(embeddings).astype(np.float32)\n')
        _modified = True
    if 'self.model.encode([query_text])\n' in _src:
        _src = _src.replace('self.model.encode([query_text])\n', 'self.model.encode([query_text]).astype(np.float32)\n')
        _modified = True
    if _modified:
        with open(_retriever_path, 'w') as f:
            f.write(_src)
        print("Patched dense_retriever.py (float32 casts for FAISS)")

    # --- Fix afrique_qwen.py on disk: avoid huge memory from output_scores=True ---
    _gen_path = os.path.join(project_root, 'generation', 'afrique_qwen.py')
    with open(_gen_path, 'r') as f:
        _gen_src = f.read()
    _gen_modified = False
    if 'output_scores=True' in _gen_src:
        _gen_src = _gen_src.replace('output_scores=True', 'output_scores=False')
        _gen_modified = True
    if _gen_modified:
        with open(_gen_path, 'w') as f:
            f.write(_gen_src)
        print("Patched afrique_qwen.py (output_scores=False)")
else:
    project_root = os.path.abspath(os.path.dirname(globals().get('__vsc_ipynb_file__', os.getcwd())))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Contents: {os.listdir(project_root)}")

# Check GPU availability
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: c:\Users\bgano\Desktop\CMU\Courses\Spring 2026\Independent Research Project\robustness of RAG for Low Resource Languages
Contents: ['.env', '.git', '.gitignore', '.vscode', 'cache', 'config', 'data', 'evaluation', 'generation', 'noise', 'pipeline', 'Readme.md', 'requirements.txt', 'retrieval', 'run_minimal.ipynb', 'scripts', 'utils']

PyTorch version: 2.10.0+cpu
CUDA available: False


## 1. Imports & Authentication


In [2]:
from dotenv import load_dotenv
from huggingface_hub import login
from data.dataset import AfriQALoader
from pipeline.rag_pipeline import RAGPipeline
from evaluation.metrics import Evaluator, contains_gold

# --- Get HF_TOKEN ---
hf_token = None

# 1) Try Colab Secrets (set via key icon in left sidebar → "HF_TOKEN")
if IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

# 2) Try .env file
if not hf_token:
    load_dotenv(os.path.join(project_root, '.env'))
    hf_token = os.environ.get("HF_TOKEN")

# 3) Manual input as last resort
if not hf_token:
    from getpass import getpass
    hf_token = getpass("Enter your HF_TOKEN: ")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("WARNING: No HF_TOKEN provided.")

Logged in to Hugging Face Hub.


## 2. Configuration

In [3]:
language = 'swa'       # Swahili
num_examples = 3

print(f"Language: {language}")
print(f"Number of examples: {num_examples}")

Language: swa
Number of examples: 3


## 3. Load Dataset

In [4]:
print(f"Loading AfriQA ({language})...")
loader = AfriQALoader()
examples = loader.load(language, split='test', num_samples=num_examples)
print(f"Loaded {len(examples)} examples.")

Loading AfriQA (swa)...
Loaded 302 examples for swa (test split)
Loaded 3 examples.


## 4. Initialize RAG Pipeline
Loading the 8B LLM (4-bit quantized, ~5 GB) + embedding model (~2 GB) + corpus index can exceed T4 memory.  
We load the retriever with `float16` and clear GPU cache between steps to stay within budget.


In [ ]:
import gc
import numpy as np
import faiss as faiss_lib

# Free any previously loaded models
torch.cuda.empty_cache()
gc.collect()

# --- Step 1: Build retriever index first ---
print("Loading retriever and indexing corpus...")
from data.wikipedia import WikipediaCorpus
from retrieval.dense_retriever import DenseRetriever

# Patch index_corpus to cast embeddings to float32 before FAISS normalize
def _patched_index_corpus(self, passages):
    from tqdm import tqdm
    self.passages = passages
    texts = [f"{self.passage_prefix}{p['text']}" for p in passages]
    batch_size = 32
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Indexing passages"):
        batch = texts[i:i+batch_size]
        with torch.no_grad():
            batch_embeddings = self.model.encode(batch, show_progress_bar=False)
        embeddings.append(batch_embeddings)
    embeddings = np.vstack(embeddings).astype(np.float32)
    faiss_lib.normalize_L2(embeddings)
    dimension = embeddings.shape[1]
    self.index = faiss_lib.IndexFlatIP(dimension)
    self.index.add(embeddings)
    print(f"  Indexed {len(passages)} passages")
DenseRetriever.index_corpus = _patched_index_corpus

# Patch retrieve to cast query embedding to float32
def _patched_retrieve(self, query, k=10):
    if self.index is None:
        raise RuntimeError("No corpus indexed. Call index_corpus() first.")
    query_text = f"{self.query_prefix}{query}"
    query_embedding = self.model.encode([query_text]).astype(np.float32)
    faiss_lib.normalize_L2(query_embedding)
    scores, indices = self.index.search(query_embedding, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        doc = self.passages[idx].copy()
        doc['score'] = float(score)
        doc['retrieval_rank'] = len(results) + 1
        results.append(doc)
    return results
DenseRetriever.retrieve = _patched_retrieve

retriever = DenseRetriever(model_name="intfloat/multilingual-e5-base")
retriever.model.half()  # fp16

corpus = WikipediaCorpus(language)
retriever.index_corpus(corpus.get_passages())

# Free corpus and move retriever to CPU
del corpus
gc.collect()
retriever.model.to('cpu')
torch.cuda.empty_cache()
gc.collect()

print(f"GPU memory after indexing: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# --- Step 2: Load the LLM (8B, 4-bit quantized ~5 GB) ---
print("\nLoading generator (AfriqueQwen-8B, 4-bit)...")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from generation.afrique_qwen import AfriqueQwenGenerator

_model_name = "McGill-NLP/AfriqueQwen-8B"
_hf = os.environ.get("HF_TOKEN")

generator = AfriqueQwenGenerator.__new__(AfriqueQwenGenerator)
generator.model_name = _model_name

print("  Loading tokenizer...")
generator.tokenizer = AutoTokenizer.from_pretrained(_model_name, token=_hf)
if generator.tokenizer.pad_token is None:
    generator.tokenizer.pad_token = generator.tokenizer.eos_token

print("  Loading model with 4-bit quantization...")
generator.model = AutoModelForCausalLM.from_pretrained(
    _model_name,
    token=_hf,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    ),
    device_map="auto",
)
generator.device = next(generator.model.parameters()).device
print(f"  Model loaded on {generator.device}")

print(f"GPU memory after LLM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# --- Step 3: Assemble pipeline manually (avoid __init__ signature issues) ---
from generation.prompts import PromptManager
import pipeline.rag_pipeline as rag_pipeline_module

# Force lower generation length to avoid OOM in generate()
rag_pipeline_module.MAX_NEW_TOKENS = 48

pipeline = RAGPipeline.__new__(RAGPipeline)
pipeline.language = language
pipeline.use_retrieval = True
pipeline.prompt_manager = PromptManager(language)
pipeline.generator = generator
pipeline.retriever = retriever

print(f"GPU memory total: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Pipeline initialized.")

Loading retriever and indexing corpus...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading cached Wikipedia for swa


Indexing passages: 100%|██████████| 284/284 [00:55<00:00,  5.08it/s]


  Indexed 9060 passages
GPU memory after indexing: 0.01 GB

Loading generator (AfriqueQwen-8B, 4-bit)...
  Loading tokenizer...
  Loading model with 4-bit quantization...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Model loaded on cuda:0
GPU memory after LLM: 6.43 GB
GPU memory total: 6.43 GB
Pipeline initialized.


## 5. Run on Examples

In [ ]:
predictions = []
golds = []

# Smaller retrieval context and shorter generation reduce VRAM pressure
k_docs = 3
max_new_tokens = 48

for i, ex in enumerate(examples):
    print(f"\n--- Example {i+1} ---")
    print(f"Q: {ex['question']}")
    print(f"Gold: {ex['translated_answer']}")

    # Optional cleanup before each generation step
    torch.cuda.empty_cache()

    # Run RAG with reduced retrieval depth
    result = pipeline.run(ex['question'], k=k_docs, return_docs=True)

    # Truncate overly long generations to keep memory stable across loop iterations
    if isinstance(result.get('answer'), str) and len(result['answer']) > 1200:
        result['answer'] = result['answer'][:1200]

    print(f"A: {result['answer']}")
    print(f"Confidence: {result['confidence']:.3f}" if result['confidence'] else "Confidence: N/A")

    # Show top retrieved doc
    if 'documents' in result and result['documents']:
        print(f"Top doc: {result['documents'][0]['text'][:100]}...")

    # Check if contains gold
    if contains_gold(result['answer'], ex['translated_answer']):
        print("Contains gold answer")
    else:
        print("Does NOT contain gold answer")

    predictions.append(result['answer'])
    golds.append(ex['translated_answer'])


--- Example 1 ---
Q: Je,nani mwanzilishi wa mtandao wa kijamii ya Facebook?
Gold: ['Mark Zuckerberg with fellow Harvard College students and roommates Eduardo Saverin, Andrew McCollum, Dustin Moskovitz, and Chris Hughes']


OutOfMemoryError: CUDA out of memory. Tried to allocate 786.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 439.81 MiB is free. Including non-PyTorch memory, this process has 14.13 GiB memory in use. Of the allocated memory 13.13 GiB is allocated by PyTorch, and 899.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 6. Evaluate

In [ ]:
evaluator = Evaluator()
results = evaluator.evaluate(predictions, golds)

print("\n=== Evaluation Results ===")
for metric, value in results.items():
    print(f"{metric}: {value:.3f}")

print("\nMinimal test completed!")